## Manufacturing Statistical Analysis

### Connect to database and read SQL table

In [ ]:
import pandas as pd
import pyodbc

In [ ]:
conn_str = (
    "Driver={ODBC Driver 17 for SQL Server};"
    "Server=localhost\\SQLEXPRESS;"
    "Database=IRLManufacturingDB;" 
    "Trusted_Connection=yes;"
)
conn = pyodbc.connect(conn_str)
query = 'SELECT * FROM gold.machine1'

df = pd.read_sql(query, conn)

### Data Exploration

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
df.info()

Observations:
1. There are no NULL values.
2. Production_shift is a category but stands as an object datatype

In [ ]:
df.head(5)

Observations:

3. **The data granularity is hourly (total machine time = 3600s)**
4. There are rows with no total production (good + bad pieces)
5. There are rows with the whole hour planned as downtime (machine_scheduled_downtime = 3600s)

Actions:

1. Delete duplicated columns (*production_datetime*)
2. Create dimension or change data-types as necessary
3. In the cases where there are not produced pieces, and where the whole hour is scheduled as downtime
is clear that we don't need this info since in the first case we can't use this info to analize production related metrics, and in the second case the whole time was planned as downtime, so there's not valuable information there.

In [ ]:
# 1.

# There are not duplicated columns.

In [ ]:
# 2.

df['production_shift'] = df['production_shift'].astype('category')
df['production_year'] = df['production_datetime'].dt.year
df['production_month'] = df['production_datetime'].dt.month

df['total_production_pieces'] = df['production_good_pieces'] + df['production_bad_pieces']
df['total_no_scheduled_downtime'] = df['machine_running_time'] + df['machine_idle_time'] + df['machine_breakdown_time']

In [ ]:
# 3.
# There are no NULL values but there are 0 values that we need to be careful with.
# Particularly where (1) there are not produced pieces at all and where 
# (2) the whole hour is scheduled as downtime.

nzero_df = df[(df['total_production_pieces'] != 0) & (df['total_no_scheduled_downtime'] != 0)]

### Exploratory Data Analysis

#### 5-number Production Summary

In [ ]:
production_cols = ['production_bad_pieces', 'total_production_pieces']

nzero_df[production_cols].describe()

In [ ]:
ax4 = sns.boxplot(nzero_df[production_cols], orient='h', palette='Set2', flierprops={"markeredgecolor": "white"})
ax4.set_facecolor("#343837")          # Inner color
ax4.figure.set_facecolor("#343837")    # Outer color
ax4.tick_params(labelcolor='white')

In [ ]:
prd_iqr = nzero_df['production_bad_pieces'].describe().loc['75%'] - nzero_df['production_bad_pieces'].describe().loc['25%']
prd_upprlim =  nzero_df['production_bad_pieces'].describe().loc['75%'] + 1.5*prd_iqr
prd_lwrlim = max(nzero_df['production_bad_pieces'].describe().loc['25%'] - 1.5*prd_iqr,0)

print(f"Interquartile Range: {prd_iqr} \tUpper Limit: {prd_upprlim} \tLower Limit: {prd_lwrlim}")

In [ ]:
nzero_df[nzero_df['production_bad_pieces'] > 69.5].drop(['production_datetime', 'production_shift'],axis=1).groupby('production_year').count()

In [ ]:
nzero_df.drop(['production_datetime', 'production_shift'],axis=1).describe()

In [ ]:
nzero_df[nzero_df['production_bad_pieces'] <= 69.5].drop(['production_datetime', 'production_shift'],axis=1).describe()

#### Production Data Distribution

In [ ]:
ax7 = sns.histplot(data=nzero_df, x="production_bad_pieces", bins=15, kde=True, edgecolor='white', line_kws={"color": 'white', "linewidth": 2.5})
ax7.set_facecolor("#343837")          # Inner color
ax7.figure.set_facecolor("#343837")    # Outer color
ax7.tick_params(labelcolor='white')
ax7.set_xlabel('PIECES', color='tomato')
ax7.set_ylabel('COUNT', color='tomato')
ax7.set_title('PRODUCED BAD PIECES', color='tomato')

In [ ]:
ax8 = sns.histplot(data=nzero_df, x="total_production_pieces", bins=15, kde=True, edgecolor='white', line_kws={"color": 'white', "linewidth": 2.5})
ax8.set_facecolor("#343837")          # Inner color
ax8.figure.set_facecolor("#343837")    # Outer color
ax8.tick_params(labelcolor='white')
ax8.set_xlabel('PIECES', color='tomato')
ax8.set_ylabel('COUNT', color='tomato')
ax8.set_title('TOTAL PRODUCED PIECES', color='tomato')

#### 5-number Machine-Time Summary

In [ ]:
machine_cols = ['machine_breakdown_time', 'total_no_scheduled_downtime']

nzero_df[machine_cols].describe()

In [ ]:
ax5 = sns.boxplot(nzero_df[machine_cols], orient='h', palette='Set2', flierprops={"markeredgecolor": "white"})
ax5.set_facecolor("#343837")          # Inner color
ax5.figure.set_facecolor("#343837")    # Outer color
ax5.tick_params(labelcolor='white')

In [ ]:
mac_iqr = nzero_df['machine_breakdown_time'].describe().loc['75%'] - nzero_df['machine_breakdown_time'].describe().loc['25%']
mac_upprlim =  nzero_df['machine_breakdown_time'].describe().loc['75%'] + 1.5*mac_iqr
mac_lwrlim = max(nzero_df['machine_breakdown_time'].describe().loc['25%'] - 1.5*mac_iqr,0)

print(f"Interquartile Range: {mac_iqr} \tUpper Limit: {mac_upprlim} \tLower Limit: {mac_lwrlim}")

In [ ]:
nzero_df[nzero_df['machine_breakdown_time'] > 357.5].drop(['production_datetime', 'production_shift'],axis=1).groupby('production_year').count()

In [ ]:
nzero_df.drop(['production_datetime', 'production_shift'],axis=1).describe()

In [ ]:
nzero_df[nzero_df['machine_breakdown_time'] <= 357.5].drop(['production_datetime', 'production_shift'],axis=1).describe()

#### Machine-time Data Distribution

In [ ]:
ax11 = sns.histplot(data=nzero_df, x="machine_breakdown_time", bins=15, kde=True, edgecolor='white', line_kws={"color": 'white', "linewidth": 2.5})
ax11.set_facecolor("#343837")          # Inner color
ax11.figure.set_facecolor("#343837")    # Outer color
ax11.tick_params(labelcolor='white')
ax11.set_xlabel('TIME (sec)', color='tomato')
ax11.set_ylabel('COUNT', color='tomato')
ax11.set_title('MACHINE BREAKDOWN TIME', color='tomato')

In [ ]:
ax12 = sns.histplot(data=nzero_df, x="total_no_scheduled_downtime", bins=15, kde=True, edgecolor='white', line_kws={"color": 'white', "linewidth": 2.5})
ax12.set_facecolor("#343837")          # Inner color
ax12.figure.set_facecolor("#343837")    # Outer color
ax12.tick_params(labelcolor='white')
ax12.set_xlabel('TIME (sec)', color='tomato')
ax12.set_ylabel('COUNT', color='tomato')
ax12.set_title('TOTAL NO-SCHEDULED DOWNTIME', color='tomato')

### Production Magnitude Analysis

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax2 = sns.boxplot(nzero_df, x='production_month', y='total_production_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax2.set_facecolor("#343837")          # Inner color
ax2.figure.set_facecolor("#343837")    # Outer color
ax2.set_xlabel('MONTH', color='tomato')
ax2.set_ylabel('TOTAL PRODUCTION', color='tomato')
ax2.tick_params(labelcolor='white')
ax2.set_title('PIECES PRODUCED BY MONTH', color='tomato')
#plt.ylim(300, 550)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax3 = sns.boxplot(nzero_df, x='production_month', y='production_bad_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax3.set_facecolor("#343837")          # Inner color
ax3.figure.set_facecolor("#343837")    # Outer color
ax3.set_xlabel('MONTH', color='tomato')
ax3.set_ylabel('TOTAL DEFECTIVE PIECES', color='tomato')
ax3.set_title('DEFECTIVE PIECES PRODUCED BY MONTH', color='tomato')
ax3.tick_params(labelcolor='white')
#plt.ylim(0, 65)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax12 = sns.boxplot(nzero_df, x='production_month', y='production_quality_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax12.set_facecolor("#343837")          # Inner color
ax12.figure.set_facecolor("#343837")    # Outer color
ax12.set_xlabel('MONTH', color='tomato')
ax12.set_ylabel('QUALITY PERCENTAGE', color='tomato')
ax12.tick_params(labelcolor='white')
ax12.set_title('QUALITY PCT BY MONTH', color='tomato')
#plt.ylim(80, 100)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax = sns.boxplot(nzero_df, x='production_shift', y='total_production_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax.set_facecolor("#343837")          # Inner color
ax.figure.set_facecolor("#343837")    # Outer color
ax.set_xlabel('SHIFT', color='tomato')
ax.set_ylabel('TOTAL PRODUCTION', color='tomato')
ax.set_title('PIECES PRODUCED BY SHIFT', color='tomato')
ax.tick_params(labelcolor='white')
#plt.ylim(300, 500)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax1 = sns.boxplot(nzero_df, x='production_shift', y='production_bad_pieces', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax1.set_facecolor("#343837")          # Inner color
ax1.figure.set_facecolor("#343837")    # Outer color
ax1.set_xlabel('SHIFT', color='tomato')
ax1.set_ylabel('TOTAL DEFECTIVE PIECES', color='tomato')
ax1.set_title('DEFECTIVE PIECES PRODUCED BY SHIFT', color='tomato')
ax1.tick_params(labelcolor='white')
#plt.ylim(20, 30)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax13 = sns.boxplot(nzero_df, x='production_shift', y='production_quality_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax13.set_facecolor("#343837")          # Inner color
ax13.figure.set_facecolor("#343837")    # Outer color
ax13.set_xlabel('SHIFT', color='tomato')
ax13.set_ylabel('QUALITY PERCENTAGE', color='tomato')
ax13.set_title('QUALITY PERCENTAGE BY SHIFT', color='tomato')
ax13.tick_params(labelcolor='white')
#plt.ylim(80, 100)

#### Time Analysis

In [ ]:
monthly_prd_bad_pcs_df = nzero_df.groupby(pd.Grouper(key='production_datetime', freq='MS'))['production_bad_pieces'].sum()

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax19 = sns.lineplot(monthly_prd_bad_pcs_df, x=monthly_prd_bad_pcs_df.index, y=monthly_prd_bad_pcs_df)

ax19.set_facecolor("#343837")          # Inner color
ax19.figure.set_facecolor("#343837")    # Outer color
ax19.set_xlabel('DATE', color='tomato')
ax19.set_ylabel('PRODUCED BAD PIECES', color='tomato')
ax19.set_title('PRODUCED BAD PIECES OVER TIME', color='tomato')
ax19.tick_params(labelcolor='white')

### Machine-time Magnitude Analysis

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax15 = sns.boxplot(nzero_df, x='production_month', y='total_no_scheduled_downtime', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax15.set_facecolor("#343837")          # Inner color
ax15.figure.set_facecolor("#343837")    # Outer color
ax15.set_xlabel('MONTH', color='tomato')
ax15.set_ylabel('TIME (sec)', color='tomato')
ax15.tick_params(labelcolor='white')
ax15.set_title('NO-SCHEDULED DOWNTIME BY MONTH', color='tomato')
#plt.ylim(300, 550)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax15 = sns.boxplot(nzero_df, x='production_month', y='machine_breakdown_time', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax15.set_facecolor("#343837")          # Inner color
ax15.figure.set_facecolor("#343837")    # Outer color
ax15.set_xlabel('MONTH', color='tomato')
ax15.set_ylabel('MACHINE BREAKDOWN TIME', color='tomato')
ax15.set_title('MACHINE BREAKDOWN TIME BY MONTH', color='tomato')
ax15.tick_params(labelcolor='white')
#plt.ylim(0, 65)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax16 = sns.boxplot(nzero_df, x='production_month', y='machine_availability_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax16.set_facecolor("#343837")          # Inner color
ax16.figure.set_facecolor("#343837")    # Outer color
ax16.set_xlabel('MONTH', color='tomato')
ax16.set_ylabel('AVAILABILITY PERCENTAGE', color='tomato')
ax16.tick_params(labelcolor='white')
ax16.set_title('AVAILABILITY PCT BY MONTH', color='tomato')
#plt.ylim(80, 100)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax17 = sns.boxplot(nzero_df, x='production_shift', y='total_no_scheduled_downtime', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax17.set_facecolor("#343837")          # Inner color
ax17.figure.set_facecolor("#343837")    # Outer color
ax17.set_xlabel('SHIFT', color='tomato')
ax17.set_ylabel('NO-SCHEDULED DOWNTIME', color='tomato')
ax17.set_title('NO-SCHEDULED DOWNTIME BY SHIFT', color='tomato')
ax17.tick_params(labelcolor='white')
#plt.ylim(300, 500)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax17 = sns.boxplot(nzero_df, x='production_shift', y='machine_breakdown_time', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax17.set_facecolor("#343837")          # Inner color
ax17.figure.set_facecolor("#343837")    # Outer color
ax17.set_xlabel('SHIFT', color='tomato')
ax17.set_ylabel('BREAKDOWN TIME', color='tomato')
ax17.set_title('BREAKDOWN TIME BY SHIFT', color='tomato')
ax17.tick_params(labelcolor='white')
#plt.ylim(300, 500)

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax18 = sns.boxplot(nzero_df, x='production_shift', y='machine_availability_percent', hue='production_year', palette='Set2', flierprops={"markeredgecolor": "white"})

ax18.set_facecolor("#343837")          # Inner color
ax18.figure.set_facecolor("#343837")    # Outer color
ax18.set_xlabel('SHIFT', color='tomato')
ax18.set_ylabel('AVAILABILITY PCT', color='tomato')
ax18.set_title('AVAILABILITY PCT BY SHIFT', color='tomato')
ax18.tick_params(labelcolor='white')
#plt.ylim(300, 500)

### Time Analysis

In [ ]:
monthly_mac_breakdown_time_df = nzero_df.groupby(pd.Grouper(key='production_datetime', freq='MS'))['machine_breakdown_time'].sum()

In [ ]:
plt.figure(figsize=(12,4), dpi=200)
ax20 = sns.lineplot(monthly_mac_breakdown_time_df, x=monthly_mac_breakdown_time_df.index, y=monthly_mac_breakdown_time_df)

ax20.set_facecolor("#343837")          # Inner color
ax20.figure.set_facecolor("#343837")    # Outer color
ax20.set_xlabel('DATE', color='tomato')
ax20.set_ylabel('BREAKDOWN TIME', color='tomato')
ax20.set_title('BREAKDOWN TIME OVER TIME', color='tomato')
ax20.tick_params(labelcolor='white')